In [1]:

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [2]:
import pandas as pd
import numpy as np

from src.feature_engineering import (
    drop_customer_id,
    split_features_target,
    encode_features,
    split_train_test,
    scale_features, 
    save_processed_data
)

In [3]:
from src.model_training import train_logistic_regression

In [4]:
data = pd.read_csv("../data/telco_customer_churn_clean.csv")

In [5]:
data = drop_customer_id(data)

X, y = split_features_target(data, "Churn")

X_encoded = encode_features(X)

In [6]:
X_train, X_test, y_train, y_test = split_train_test(
    X_encoded,
    y
)

In [7]:
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train,
    X_test
)

In [8]:
logistic_model = train_logistic_regression(
    X_train_scaled,
    y_train
)

In [9]:
type(logistic_model)

sklearn.linear_model._logistic.LogisticRegression

In [10]:
print(logistic_model)

LogisticRegression(max_iter=1000, random_state=42)


In [11]:
y_pred = logistic_model.predict(X_test_scaled)

In [12]:
print(y_pred[:10])

['No' 'Yes' 'No' 'No' 'No' 'Yes' 'No' 'No' 'No' 'No']


In [13]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(10)

,Actual,Predicted
0,No,No
1,No,Yes
2,No,No
3,No,No
4,No,No
5,No,Yes
6,No,No
7,No,No
8,No,No
9,Yes,No


In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[925 110]
 [162 212]]


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="Yes"))
print("Recall   :", recall_score(y_test, y_pred, pos_label="Yes"))
print("F1 Score :", f1_score(y_test, y_pred, pos_label="Yes"))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8069552874378992
Precision: 0.6583850931677019
Recall   : 0.5668449197860963
F1 Score : 0.6091954022988506

Classification Report
              precision    recall  f1-score   support

          No       0.85      0.89      0.87      1035
         Yes       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [16]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": logistic_model.coef_[0]
})

feature_importance["Absolute"] = feature_importance["Coefficient"].abs()

feature_importance = feature_importance.sort_values(
    by="Absolute",
    ascending=False
)

feature_importance.head(15)

,Feature,Coefficient,Absolute
1,tenure,-1.236528,1.236528
2,MonthlyCharges,-0.920153,0.920153
10,InternetService_Fiber optic,0.776154,0.776154
25,Contract_Two year,-0.586859,0.586859
3,TotalCharges,0.514285,0.514285
24,Contract_One year,-0.285509,0.285509
23,StreamingMovies_Yes,0.257227,0.257227
21,StreamingTV_Yes,0.257144,0.257144
9,MultipleLines_Yes,0.216167,0.216167
26,PaperlessBilling_Yes,0.182034,0.182034


## Random Forest ##

In [17]:
from src.model_training import train_random_forest

random_forest_model = train_random_forest(
    X_train_scaled,
    y_train
)

In [19]:
rf_pred = random_forest_model.predict(X_test_scaled)

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred, pos_label="Yes"))
print("Recall   :", recall_score(y_test, rf_pred, pos_label="Yes"))
print("F1 Score :", f1_score(y_test, rf_pred, pos_label="Yes"))

Accuracy : 0.7849538679914834
Precision: 0.6187290969899666
Recall   : 0.4946524064171123
F1 Score : 0.549777117384844


## Comparing Both Models


In [21]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label="Yes"),
        recall_score(y_test, y_pred, pos_label="Yes"),
        f1_score(y_test, y_pred, pos_label="Yes")
    ],
    "Random Forest": [
        accuracy_score(y_test, rf_pred),
        precision_score(y_test, rf_pred, pos_label="Yes"),
        recall_score(y_test, rf_pred, pos_label="Yes"),
        f1_score(y_test, rf_pred, pos_label="Yes")
    ]
})

comparison

,Metric,Logistic Regression,Random Forest
0,Accuracy,0.806955,0.784954
1,Precision,0.658385,0.618729
2,Recall,0.566845,0.494652
3,F1 Score,0.609195,0.549777


In [23]:
from src.model_training import save_model

save_model(
    logistic_model,
    "../models/logistic_regression.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [24]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler Saved Successfully!")

Scaler Saved Successfully!


In [25]:
joblib.dump(
    X_train.columns.tolist(),
    "../models/feature_columns.pkl"
)

print("Feature Columns Saved!")

Feature Columns Saved!
